In [1]:
%cd ..

c:\Users\micha\OneDrive\Desktop\Python_Workspace\BikeBuildPartTracker\src


In [2]:
from dataclasses import dataclass
from enum import Enum
from bs4 import BeautifulSoup, Tag
from Clients.BaseWebClient import BaseWebClient
from DTOs.PriceDTO import PriceDTO
from datetime import date

In [3]:
@dataclass(frozen=True)
class CondorSku:
    """A single sellable variant on condorcycles.com."""
    product_slug: str
    variant_id: int

    @property
    def url(self) -> str:
        return (
            f"https://www.condorcycles.com/en-us/products/"
            f"condor-{self.product_slug}.json"
        )
    # https://www.condorcycles.com/en-us/products/condor-fratello-disc-thru-axle-frameset.json
    
    @property
    def product_name(self) -> str:
        return (self.product_slug.replace('-', ' '))


class CondorProductSkus(Enum):
    BLUE_CONDOR_FRAME = CondorSku("fratello-disc-thru-axle-frameset", 49351522615617)
    RED_CONDOR_FRAME = CondorSku("fratello-disc-thru-axle-frameset", 55310658929024)
    TPU_TUBE_BUNDLE = CondorSku("tpu-inner-tube-bundle", 56942405157248)
    TPU_REPAIR_KIT = CondorSku("tpu-inner-tube-patch-repair-kit", 56434152472960)


class CondorWebClient(BaseWebClient):
    def __init__(self, timeout = 10):
        super().__init__(timeout)

    def _fetch(self, url: str) -> dict:
        resp = self.session.get(url, timeout=self.timeout)
        resp.raise_for_status()
        return resp.json()

    def _parse(self, product_json: dict, condor_product_sku: CondorSku) -> dict:
        variants = product_json["product"]["variants"]
        variant = next(v for v in variants if v["id"] == condor_product_sku.variant_id)
        
        price = float(variant["price"])
        compare_at = variant.get("compare_at_price")
        
        if compare_at and float(compare_at) > price:
            msrp_price, sale_price = float(compare_at), price
        else:
            msrp_price, sale_price = price, None
        
        return {
            "product_id": condor_product_sku.product_slug,
            "name": condor_product_sku.product_name,
            "date": date.today(),
            "msrp_price": msrp_price,
            "sale_price": sale_price,
        }

In [4]:
condor_client = CondorWebClient()

In [5]:
condor_red_frame_dto: PriceDTO = condor_client.fetch_price_dto(CondorProductSkus.RED_CONDOR_FRAME.value)
print(condor_red_frame_dto)

PriceDTO(product_id='fratello-disc-thru-axle-frameset', name='fratello disc thru axle frameset', date=datetime.date(2026, 7, 21), msrp_price=2300.0, sale_price=None, currency='USD')


In [6]:
condor_blue_frame_dto: PriceDTO = condor_client.fetch_price_dto(CondorProductSkus.BLUE_CONDOR_FRAME.value)
print(condor_blue_frame_dto)

PriceDTO(product_id='fratello-disc-thru-axle-frameset', name='fratello disc thru axle frameset', date=datetime.date(2026, 7, 21), msrp_price=2250.0, sale_price=1900.0, currency='USD')


In [7]:
condor_tpu_bundle_dto: PriceDTO = condor_client.fetch_price_dto(CondorProductSkus.TPU_TUBE_BUNDLE.value)
print(condor_tpu_bundle_dto)

PriceDTO(product_id='tpu-inner-tube-bundle', name='tpu inner tube bundle', date=datetime.date(2026, 7, 21), msrp_price=80.0, sale_price=68.0, currency='USD')


In [8]:
condor_tpu_repair_kit_dto: PriceDTO = condor_client.fetch_price_dto(CondorProductSkus.TPU_REPAIR_KIT.value)
print(condor_tpu_repair_kit_dto)

PriceDTO(product_id='tpu-inner-tube-patch-repair-kit', name='tpu inner tube patch repair kit', date=datetime.date(2026, 7, 21), msrp_price=7.99, sale_price=None, currency='USD')


In [9]:
@dataclass(frozen=True)
class WhiteIndustriesSku:
    """A single sellable product on whiteind.com."""
    product_slug: str

    @property
    def url(self) -> str:
        return f"https://www.whiteind.com/product/{self.product_slug}/"

    @property
    def product_name(self) -> str:
        return self.product_slug.replace('-', ' ')


class WhiteIndustriesProductSkus(Enum):
    CLD_ALUMINUM_700C_ROAD_WHEELS = WhiteIndustriesSku("cld-aluminum-700c-road-wheels")


class WhiteIndustriesWebClient(BaseWebClient):
    def __init__(self, timeout: int = 10):
        super().__init__(timeout)

    def _parse(self, html_soup: BeautifulSoup, wi_sku: WhiteIndustriesSku) -> dict[str, str | float]:
        msrp_price, sale_price = self._parse_html_price_prices(html_soup)
        return {
            "product_id": wi_sku.product_slug,
            "name": wi_sku.product_name,
            "date": date.today(),
            "msrp_price": msrp_price,
            "sale_price": sale_price,
        }

    def _parse_html_price_prices(self, html_soup: BeautifulSoup) -> tuple[float, float | None]:
        # First match = main product price block; related-product price
        # blocks appear later in the DOM, so this is safe without extra scoping.
        price_block = html_soup.select_one('div[data-block-name="woocommerce/product-price"]')
        if not price_block:
            raise Exception("Error: Product price block not found on page.")

        del_tag: Tag | None = price_block.select_one("del .woocommerce-Price-amount")
        ins_tag: Tag | None = price_block.select_one("ins .woocommerce-Price-amount")

        if del_tag and ins_tag:
            msrp_price = self._to_float(del_tag.get_text(strip=True))
            sale_price = self._to_float(ins_tag.get_text(strip=True))
        else:
            amount_tag: Tag | None = price_block.select_one(".woocommerce-Price-amount")
            if not amount_tag:
                raise Exception("Error: No MSRP or Sale price could be found.")
            msrp_price = self._to_float(amount_tag.get_text(strip=True))
            sale_price = None

        return msrp_price, sale_price

    @staticmethod
    def _to_float(text: str) -> float:
        return float(text.lstrip('$').replace(',', ''))

In [10]:
white_industries_client = WhiteIndustriesWebClient()

In [11]:
white_industries_dto: PriceDTO = white_industries_client.fetch_price_dto(WhiteIndustriesProductSkus.CLD_ALUMINUM_700C_ROAD_WHEELS.value)
print(white_industries_dto)

PriceDTO(product_id='cld-aluminum-700c-road-wheels', name='cld aluminum 700c road wheels', date=datetime.date(2026, 7, 21), msrp_price=1338.0, sale_price=None, currency='USD')


In [12]:
@dataclass(frozen=True)
class FullSpeedAheadSku:
    """A single sellable product from Full Speed Ahead."""
    product_slug: str
    variant_id: int

    @property
    def url(self) -> str:
        return f"https://www.fsaproshop.com/products/{self.product_slug}.json"

    @property
    def product_name(self) -> str:
        return self.product_slug.replace('-', ' ')



class FullSpeedAheadSkus(Enum):
    _ENERGY_HANDLEBAR_38CM_VARIANT_ID = 32882646679629
    ENERGY_SUPER_COMPACT_HANDLEBAR = FullSpeedAheadSku("energy-super-compact-handlebar", _ENERGY_HANDLEBAR_38CM_VARIANT_ID)
    
    _OMEGA_STEM_90MM_VARIANT_ID = 46394515456190
    OMEGA_STEM = FullSpeedAheadSku("omega-stem", _OMEGA_STEM_90MM_VARIANT_ID)
    
    _ENERGY_STEM_90MM_VARIANT_ID = 46395174158526
    ENERGY_STEM = FullSpeedAheadSku("energy-stem", _ENERGY_STEM_90MM_VARIANT_ID)
    
    _HEADSET_SPACER_KIT_VARIANT_ID = 32835022389325
    HEADSET_SPACER_KIT = FullSpeedAheadSku("headset-spacer-kit-w-fsa-logo-assorted-sizes-1", _HEADSET_SPACER_KIT_VARIANT_ID)


class FullSpeedAheadWebClient(BaseWebClient):
    def __init__(self, timeout: int = 10):
        super().__init__(timeout)

    def _fetch(self, url: str) -> dict:
        resp = self.session.get(url, timeout=self.timeout)
        resp.raise_for_status()
        return resp.json()
    
    def _parse(self, product_json: dict, fsa_product_sku: CondorSku) -> dict:
        variants = product_json["product"]["variants"]
        variant = next(v for v in variants if v["id"] == fsa_product_sku.variant_id)
        
        price = float(variant["price"])
        compare_at = variant.get("compare_at_price")
        
        if compare_at and float(compare_at) > price:
            msrp_price, sale_price = float(compare_at), price
        else:
            msrp_price, sale_price = price, None
        
        return {
            "product_id": fsa_product_sku.product_slug,
            "name": fsa_product_sku.product_name,
            "date": date.today(),
            "msrp_price": msrp_price,
            "sale_price": sale_price,
        }

In [13]:
fsa_client = FullSpeedAheadWebClient()

In [14]:
energy_super_compact_handlebar = fsa_client.fetch_price_dto(FullSpeedAheadSkus.ENERGY_SUPER_COMPACT_HANDLEBAR.value)
print(energy_super_compact_handlebar)

PriceDTO(product_id='energy-super-compact-handlebar', name='energy super compact handlebar', date=datetime.date(2026, 7, 21), msrp_price=119.0, sale_price=89.25, currency='USD')


In [15]:
omega_stem_dto = fsa_client.fetch_price_dto(FullSpeedAheadSkus.OMEGA_STEM.value)
print(omega_stem_dto)

PriceDTO(product_id='omega-stem', name='omega stem', date=datetime.date(2026, 7, 21), msrp_price=40.0, sale_price=30.0, currency='USD')


In [16]:
energy_stem_dto = fsa_client.fetch_price_dto(FullSpeedAheadSkus.ENERGY_STEM.value)
print(energy_stem_dto)

PriceDTO(product_id='energy-stem', name='energy stem', date=datetime.date(2026, 7, 21), msrp_price=129.0, sale_price=96.75, currency='USD')


In [17]:
space_kit_dto = fsa_client.fetch_price_dto(FullSpeedAheadSkus.HEADSET_SPACER_KIT.value)
print(space_kit_dto)

PriceDTO(product_id='headset-spacer-kit-w-fsa-logo-assorted-sizes-1', name='headset spacer kit w fsa logo assorted sizes 1', date=datetime.date(2026, 7, 21), msrp_price=22.0, sale_price=16.5, currency='USD')


In [18]:
@dataclass(frozen=True)
class MerlinCyclesProductSku:
    """A single sellable product from Full Speed Ahead."""
    product_slug: str

    @property
    def url(self) -> str:
        return f"https://www.merlincycles.com/en-us/{self.product_slug}.html"

    @property
    def product_name(self) -> str:
        return self.product_slug.replace('-', ' ')

class MerlinCyclesSkus(Enum):
    SHIMANO_105_MECHANICAL_GROUPSET = MerlinCyclesProductSku("shimano-105-r7120-disc-groupset-12-speed-298406")
    SHIMANO_105_DI2_GROUPSET = MerlinCyclesProductSku("shimano-105-r7170-di2-disc-groupset-12-speed-271682")
    SHIMANO_ULTEGRA_DI2_GROUPSET = MerlinCyclesProductSku("shimano-ultegra-r8170-di2-disc-groupset-12-speed-252965")

In [ ]:
from playwright.sync_api import sync_playwright
from playwright.async_api import async_playwright

@dataclass(frozen=True)
class MerlinCyclesProductSku:
    """A single sellable product from Full Speed Ahead."""
    product_slug: str

    @property
    def url(self) -> str:
        return f"https://www.merlincycles.com/en-us/{self.product_slug}.html"

    @property
    def product_name(self) -> str:
        return self.product_slug.replace('-', ' ')

class MerlinCyclesSkus(Enum):
    SHIMANO_105_MECHANICAL_GROUPSET = MerlinCyclesProductSku("shimano-105-r7120-disc-groupset-12-speed-298406")
    SHIMANO_105_DI2_GROUPSET = MerlinCyclesProductSku("shimano-105-r7170-di2-disc-groupset-12-speed-271682")
    SHIMANO_ULTEGRA_DI2_GROUPSET = MerlinCyclesProductSku("shimano-ultegra-r8170-di2-disc-groupset-12-speed-252965")

class MerlinCyclesWebClient(BaseWebClient):
    def __init__(self, timeout: int = 10):
        super().__init__(timeout)

    async def fetch_price_dto(self, sku_item) -> PriceDTO:
        html_soup = await self._fetch(sku_item.url)
        raw_data = self._parse(html_soup, sku_item)
        return self._build_dto(raw_data)
    
    async def _fetch(self, url: str) -> BeautifulSoup:
        async with async_playwright() as p:
            browser = await p.chromium.launch()
            page = await browser.new_page()
            await page.goto(url, wait_until="networkidle")
            await page.wait_for_selector("div.product-pricing", timeout=10000)
            html = await page.content()
            await browser.close()
        return BeautifulSoup(html, "html.parser")

    def _parse(self, html_soup: BeautifulSoup, merlin_product_sku) -> dict:
        msrp_price, sale_price = self._parse_html_price_prices(html_soup)
        return {
            "product_id": merlin_product_sku.product_slug,
            "name": merlin_product_sku.product_name,
            "date": date.today(),
            "msrp_price": msrp_price,
            "sale_price": sale_price,
        }

    def _parse_html_price_prices(self, html_soup: BeautifulSoup) -> tuple[float, float | None]:
        pricing = html_soup.select_one("div.product-pricing")
        if not pricing:
            raise Exception("Error: Product pricing container not found on page.")

        current_tag = pricing.select_one("div.merlin-price span.price")
        if not current_tag:
            raise Exception("Error: No current price could be found.")
        current_price = self._to_float(current_tag.get_text(strip=True))

        rrp_tag = pricing.select_one("div.product-savings span.rrp span.price")
        if rrp_tag:
            rrp_price = self._to_float(rrp_tag.get_text(strip=True))
            # RRP only counts as MSRP if it's actually higher (defensive,
            # in case of a data glitch where rrp == current)
            if rrp_price > current_price:
                return rrp_price, current_price

        return current_price, None

    @staticmethod
    def _to_float(text: str) -> float:
        return float(text.lstrip("$").replace(",", ""))
    
merlin_web_client = MerlinCyclesWebClient()
shimano_105_dto = await merlin_web_client.fetch_price_dto(MerlinCyclesSkus.SHIMANO_105_MECHANICAL_GROUPSET.value)
print(shimano_105_dto)



NotImplementedError: 

In [ ]:
@dataclass(frozen=True)
class MerlinCyclesProductSku:
    """A single sellable product from Full Speed Ahead."""
    product_slug: str

    @property
    def url(self) -> str:
        return f"https://www.merlincycles.com/en-us/{self.product_slug}.html"

    @property
    def product_name(self) -> str:
        return self.product_slug.replace('-', ' ')

class MerlinCyclesSkus(Enum):
    SHIMANO_105_MECHANICAL_GROUPSET = MerlinCyclesProductSku("shimano-105-r7120-disc-groupset-12-speed-298406")
    SHIMANO_105_DI2_GROUPSET = MerlinCyclesProductSku("shimano-105-r7170-di2-disc-groupset-12-speed-271682")
    SHIMANO_ULTEGRA_DI2_GROUPSET = MerlinCyclesProductSku("shimano-ultegra-r8170-di2-disc-groupset-12-speed-252965")

class MerlinCyclesWebClient(BaseWebClient):
    def __init__(self, timeout: int = 10):
        super().__init__(timeout)

    def _parse(self, html_soup: BeautifulSoup, merlin_product_sku) -> dict:
        msrp_price, sale_price = self._parse_html_price_prices(html_soup)
        return {
            "product_id": merlin_product_sku.product_slug,
            "name": merlin_product_sku.product_name,
            "date": date.today(),
            "msrp_price": msrp_price,
            "sale_price": sale_price,
        }

    def _parse_html_price_prices(self, html_soup: BeautifulSoup) -> tuple[float, float | None]:
        pricing = html_soup.select_one("div.product-pricing")
        if not pricing:
            raise Exception("Error: Product pricing container not found on page.")

        current_tag = pricing.select_one("div.merlin-price span.price")
        if not current_tag:
            raise Exception("Error: No current price could be found.")
        current_price = self._to_float(current_tag.get_text(strip=True))

        rrp_tag = pricing.select_one("div.product-savings span.rrp span.price")
        if rrp_tag:
            rrp_price = self._to_float(rrp_tag.get_text(strip=True))
            # RRP only counts as MSRP if it's actually higher (defensive,
            # in case of a data glitch where rrp == current)
            if rrp_price > current_price:
                return rrp_price, current_price

        return current_price, None

    @staticmethod
    def _to_float(text: str) -> float:
        return float(text.lstrip("$").replace(",", ""))
    
merlin_web_client = MerlinCyclesWebClient()
shimano_105_dto = merlin_web_client.fetch_price_dto(MerlinCyclesSkus.SHIMANO_105_MECHANICAL_GROUPSET.value)
print(shimano_105_dto)

In [ ]:
import json

@dataclass(frozen=True)
class RitcheySku:
    """A single sellable product variant on ritcheylogic.com.

    product_path is the full URL path segment (varies by category:
    e.g. 'bike/frames/p-29er-frame' vs 'product/wcs-carbon-mountain-adventure-fork'),
    since Ritchey doesn't use one consistent prefix across product types.
    """
    product_path: str
    sku: str

    @property
    def url(self) -> str:
        return f"https://ritcheylogic.com/{self.product_path}?sku={self.sku}"

    @property
    def product_name(self) -> str:
        return self.product_path.rsplit("/", 1)[-1].replace('-', ' ')

seatpost_sku = RitcheySku(product_path='bike/seatposts/comp-zero-seatpost', sku='41035317055')
test_sale_frame_sku = RitcheySku(product_path='bike/frames/p-29er-frame', sku='97452007032')

class RitcheyWebClient(BaseWebClient):
    def __init__(self, timeout: int = 10):
        super().__init__(timeout)

    def _parse(self, html_soup: BeautifulSoup, ritchey_sku: RitcheySku) -> dict:
        product_json = self._find_product_jsonld(html_soup)
        variant = self._find_variant(product_json, ritchey_sku.sku)
        current_price = float(variant["offers"]["price"])

        msrp_price, sale_price = self._resolve_prices(html_soup, current_price)

        return {
            "product_id": ritchey_sku.product_path,
            "name": variant.get("name", ritchey_sku.product_name),
            "date": date.today(),
            "msrp_price": msrp_price,
            "sale_price": sale_price,
        }

    def _find_product_jsonld(self, html_soup: BeautifulSoup) -> dict:
        for script in html_soup.select('script[type="application/ld+json"]'):
            try:
                data = json.loads(script.string)
            except (json.JSONDecodeError, TypeError):
                continue
            if isinstance(data, dict) and data.get("@type") == "Product":
                return data
        raise Exception("Error: No Product JSON-LD block found on page.")

    def _find_variant(self, product_json: dict, sku: str) -> dict:
        if product_json.get("sku") == sku:
            return product_json
        for variant in product_json.get("isVariantOf", {}).get("hasVariant", []):
            if variant.get("sku") == sku:
                return variant
        raise Exception(f"Error: SKU {sku} not found among product variants.")

    def _resolve_prices(self, html_soup: BeautifulSoup, current_price: float) -> tuple[float, float | None]:
        previous_price_tag = html_soup.select_one('span[aria-label="Previous price"]')
        if previous_price_tag:
            msrp_price = self._to_float(previous_price_tag.get_text(strip=True))
            return msrp_price, current_price
        return current_price, None

    @staticmethod
    def _to_float(text: str) -> float:
        return float(text.lstrip("$").replace(",", ""))
    
ritchery_client = RitcheyWebClient()
seatpost_dto = ritchery_client.fetch_price_dto(seatpost_sku)
print(seatpost_dto)